# LeetCode 1530 — Number of Good Leaf Nodes Pairs bằng Python

Link đề: https://leetcode.com/problems/number-of-good-leaf-nodes-pairs/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree, DP on Trees

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def countPairs(self, root: TreeNode | None, distance: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Tự dựng `TreeNode` và hàm dựng cây từ mảng level-order (có `None`) để chạy thử bài cây ngoài LeetCode.
2. Tính khoảng cách giữa hai nút bằng **tổ tiên chung thấp nhất (LCA)**: `dist(a, b) = depth[a] + depth[b] − 2·depth[lca]` — lời giải baseline so mọi cặp lá.
3. Biến cây thành **đồ thị vô hướng** và chạy **BFS giới hạn độ sâu** từ mỗi lá (đúng theo gợi ý của đề).
4. Nắm kỹ thuật **DP trên cây (post-order)**: mỗi nút trả về mảng "số lá ở độ sâu `d` bên dưới", gộp hai con để đếm cặp **có LCA chính là nút đó**.
5. Phân tích độ phức tạp `O(n · D²)` và tối ưu thành `O(n · D)` bằng tổng tiền tố, với `D = distance ≤ 10`.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân và số nguyên `distance`. Một cặp **hai lá khác nhau** được gọi là **tốt** nếu độ dài đường đi ngắn nhất (số cạnh) giữa chúng **≤ distance**. Trả về số cặp lá tốt.

**Ví dụ 1:** `root = [1,2,3,null,4]`, `distance = 3` → `1`. Các lá là `3` và `4`, đường đi `4 → 2 → 1 → 3` dài 3 ≤ 3.

**Ví dụ 2:** `root = [1,2,3,4,5,6,7]`, `distance = 3` → `2`. Cặp tốt `[4,5]` và `[6,7]` (dài 2). Cặp `[4,6]` dài 4 nên không tốt.

**Ví dụ 3:** `root = [7,1,4,6,null,5,3,null,null,null,null,null,2]`, `distance = 3` → `1`. Cặp tốt duy nhất là `[2,5]`.

**Ràng buộc:** số nút trong `[1, 2¹⁰]`; `1 ≤ Node.val ≤ 100`; `1 ≤ distance ≤ 10`.

Lưu ý: giá trị nút **có thể trùng nhau** — đừng dùng `val` làm khoá định danh nút.

In [1]:
import random
import time
from collections import deque, defaultdict
from typing import List, Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1
    return root


def to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Chuyển cây về mảng level-order (bỏ None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        q.append(node.left)
        q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 2, 3, None, 4], 3, 1),
    ([1, 2, 3, 4, 5, 6, 7], 3, 2),
    ([7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2], 3, 1),
]
for vals, d, want in examples:
    print(to_list(build_tree(vals)), "distance =", d, "-> mong đợi", want)

[1, 2, 3, None, 4] distance = 3 -> mong đợi 1
[1, 2, 3, 4, 5, 6, 7] distance = 3 -> mong đợi 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] distance = 3 -> mong đợi 1


In cây dạng "nằm ngang" để dễ nhìn (cây con phải ở trên, cây con trái ở dưới, mỗi mức thụt vào 4 ký tự):

In [2]:
def show(node, level=0):
    if node is None:
        return
    show(node.right, level + 1)
    leaf = node.left is None and node.right is None
    print("    " * level + str(node.val) + (" (lá)" if leaf else ""))
    show(node.left, level + 1)

show(build_tree(examples[2][0]))

            2 (lá)
        3
    4
        5 (lá)
7
    1
        6 (lá)


## 3. Kiến thức nền

### 3.1 Duyệt cây đệ quy & nhận biết lá

Lá là nút **không có con nào**: `node.left is None and node.right is None`. Duyệt đệ quy: xử lý con trái, con phải, rồi đến nút (post-order) — rất hợp khi kết quả của nút phụ thuộc vào kết quả của các con.

In [3]:
def leaves(node):
    if node is None:
        return []
    if node.left is None and node.right is None:
        return [node.val]
    return leaves(node.left) + leaves(node.right)

print("lá của ví dụ 3:", leaves(build_tree(examples[2][0])))

lá của ví dụ 3: [6, 5, 2]


### 3.2 Độ sâu và cha của mỗi nút

Lưu `parent[node]` và `depth[node]` trong `dict` với **khoá là chính đối tượng nút** (Python băm theo `id` mặc định), không dùng `val` vì có thể trùng.

In [4]:
def parent_depth(root):
    parent, depth = {root: None}, {root: 0}
    stack = [root]
    while stack:
        u = stack.pop()
        for v in (u.left, u.right):
            if v is not None:
                parent[v] = u
                depth[v] = depth[u] + 1
                stack.append(v)
    return parent, depth

r = build_tree([1, 2, 3, None, 4])
par, dep = parent_depth(r)
print({n.val: (par[n].val if par[n] else None, dep[n]) for n in dep})

{1: (None, 0), 2: (1, 1), 3: (1, 1), 4: (2, 2)}


### 3.3 Tổ tiên chung thấp nhất (LCA) và khoảng cách

Đưa nút sâu hơn lên cùng độ sâu, rồi cùng nhau đi lên tới khi gặp nhau. Khoảng cách = số bước đã đi của cả hai.

In [5]:
def dist_by_lca(a, b, parent, depth):
    steps = 0
    while depth[a] > depth[b]:
        a, steps = parent[a], steps + 1
    while depth[b] > depth[a]:
        b, steps = parent[b], steps + 1
    while a is not b:
        a, b, steps = parent[a], parent[b], steps + 2
    return steps

n3, n4 = r.right, r.left.right
print("dist(3, 4) =", dist_by_lca(n3, n4, par, dep))

dist(3, 4) = 3


### 3.4 BFS trên đồ thị với giới hạn số bước

Hàng đợi `deque` + tập `seen`. Mỗi phần tử trong hàng đợi kèm số bước; không mở rộng khi số bước đã bằng giới hạn.

In [6]:
graph = {0: [1, 2], 1: [0, 3], 2: [0], 3: [1]}
def bfs_limited(start, limit):
    seen, q, reach = {start}, deque([(start, 0)]), []
    while q:
        u, d = q.popleft()
        reach.append((u, d))
        if d == limit:
            continue
        for v in graph[u]:
            if v not in seen:
                seen.add(v)
                q.append((v, d + 1))
    return reach

print(bfs_limited(3, 2))

[(3, 0), (1, 1), (0, 2)]


### 3.5 Mảng đếm theo độ sâu và cộng mảng

Ta sẽ biểu diễn "các lá dưới một nút" bằng mảng `cnt` độ dài `D + 1`, `cnt[d]` = số lá cách nút đúng `d` cạnh. Khi đi lên một mức, mọi khoảng cách tăng 1: **dịch mảng sang phải một ô** và bỏ phần vượt `D`.

In [7]:
D = 4
cnt = [0, 1, 0, 2, 1]               # lá ở khoảng cách 1, 3, 3, 4
shifted = [0] + cnt[:-1]            # đi lên cha: khoảng cách +1, bỏ ô d = 5 > D
print("cnt     =", cnt)
print("shifted =", shifted)

cnt     = [0, 1, 0, 2, 1]
shifted = [0, 0, 1, 0, 2]


## 4. Ý tưởng nền tảng / trực giác

Mọi đường đi giữa hai lá `a`, `b` đều đi **lên** tới LCA rồi **xuống**: `dist = da + db` với `da`, `db` là khoảng cách từ LCA xuống mỗi lá. Và `a`, `b` nằm ở **hai cây con khác nhau** của LCA.

Tính tay ví dụ 2 (`[1,2,3,4,5,6,7]`, `D = 3`):

- Tại nút 2: con trái có lá 4 ở khoảng cách 1, con phải có lá 5 ở khoảng cách 1 → cặp (4,5) dài `1 + 1 = 2 ≤ 3` ✔.
- Tại nút 3: tương tự cặp (6,7) dài 2 ✔.
- Tại nút 1: bên trái có lá 4, 5 ở khoảng cách 2; bên phải có 6, 7 ở khoảng cách 2 → mỗi cặp dài `2 + 2 = 4 > 3` ✘.

Tổng = 2. Mỗi cặp được đếm **đúng một lần** — tại LCA của nó. Đây chính là cách 3 (DFS mảng đếm).

In [8]:
root2 = build_tree([1, 2, 3, 4, 5, 6, 7])
par2, dep2 = parent_depth(root2)
lv = [n for n in dep2 if n.left is None and n.right is None]
for i in range(len(lv)):
    for j in range(i + 1, len(lv)):
        d = dist_by_lca(lv[i], lv[j], par2, dep2)
        print(f"({lv[i].val},{lv[j].val}) dist={d} {'tốt' if d <= 3 else ''}")

(6,7) dist=2 tốt
(6,4) dist=4 
(6,5) dist=4 
(7,4) dist=4 
(7,5) dist=4 
(4,5) dist=2 tốt


## 5. Cách 1 — Brute Force / baseline: so mọi cặp lá bằng LCA

1. Một lần duyệt lấy `parent`, `depth` của mọi nút và danh sách lá.
2. Với mỗi cặp lá, tính khoảng cách bằng cách leo lên tới LCA; đếm cặp có khoảng cách ≤ `distance`.

In [9]:
def brute_lca(root, distance):
    parent, depth = parent_depth(root)
    lv = [u for u in depth if u.left is None and u.right is None]
    ans = 0
    for i in range(len(lv)):
        for j in range(i + 1, len(lv)):
            if dist_by_lca(lv[i], lv[j], parent, depth) <= distance:
                ans += 1
    return ans

for vals, d, want in examples:
    print(vals, d, "->", brute_lca(build_tree(vals), d), "mong đợi", want)

[1, 2, 3, None, 4] 3 -> 1 mong đợi 1
[1, 2, 3, 4, 5, 6, 7] 3 -> 2 mong đợi 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 -> 1 mong đợi 1


**Độ phức tạp:** gọi `L` là số lá, `h` là chiều cao. Có `L²/2` cặp, mỗi cặp leo tối đa `O(h)` → `O(L² · h)`. Với `n = 1024`: cây đầy đủ có `L = 512`, `h = 10` → ~1.3 triệu bước (vẫn chạy được trên LeetCode); nhưng cây "lệch" có thể làm `h` lớn. Space `O(n)`.

**Nút thắt:** ta tính khoảng cách cho **mọi** cặp, kể cả những cặp cách nhau rất xa, trong khi `distance ≤ 10` rất nhỏ. Ta muốn chỉ "nhìn" trong bán kính `distance`.

### Cách 1b — BFS giới hạn từ mỗi lá (theo gợi ý của đề)

Chuyển cây thành đồ thị vô hướng (mỗi nút nối với cha và con). Từ mỗi lá, BFS tối đa `distance` bước, đếm các lá **khác** gặp được. Mỗi cặp bị đếm hai lần (từ hai đầu) → chia 2.

In [10]:
def bfs_from_leaves(root, distance):
    adj = defaultdict(list)
    stack, leaves_ = [root], []
    while stack:
        u = stack.pop()
        if u.left is None and u.right is None:
            leaves_.append(u)
        for v in (u.left, u.right):
            if v is not None:
                adj[u].append(v)
                adj[v].append(u)
                stack.append(v)
    total = 0
    for s in leaves_:
        seen, q = {s}, deque([(s, 0)])
        while q:
            u, d = q.popleft()
            if u is not s and u.left is None and u.right is None:
                total += 1
            if d == distance:
                continue
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    q.append((v, d + 1))
    return total // 2

for vals, d, want in examples:
    print(vals, d, "->", bfs_from_leaves(build_tree(vals), d), "mong đợi", want)

[1, 2, 3, None, 4] 3 -> 1 mong đợi 1
[1, 2, 3, 4, 5, 6, 7] 3 -> 2 mong đợi 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 -> 1 mong đợi 1


**Độ phức tạp:** mỗi BFS chỉ thăm các nút trong bán kính `D`, số nút đó tối đa ~`3·2^(D−1)` (đi lên/xuống), nhưng bị chặn bởi `n` → `O(L · min(n, 2^D))`. Với `D = 10`, `n = 1024` có thể gần `O(L · n)`. Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đếm mỗi cặp tại **LCA** của nó. Tại nút `u`, nếu biết

- `L[i]` = số lá trong cây con trái cách `u` đúng `i` cạnh,
- `R[j]` = số lá trong cây con phải cách `u` đúng `j` cạnh,

thì số cặp tốt có LCA là `u` = `Σ L[i] · R[j]` với `i + j ≤ distance`.

**Chỉ cần nhớ khoảng cách 1..D:** lá cách `u` hơn `D` thì không thể tạo cặp tốt qua `u` hay qua bất kỳ tổ tiên nào của `u` (đi lên chỉ làm khoảng cách tăng). ⇒ mảng độ dài `D + 1` là đủ.

**Quy tắc DP (post-order):**
- Nếu `u` là `None` thì trả về mảng toàn 0.
- Nếu `u` là lá thì trả về mảng có `cnt[0] = 1` (một lá cách chính nó 0 cạnh) — hoặc theo quy ước "khoảng cách tới cha" thì `cnt[1] = 1`.
- Ngược lại: lấy `L`, `R` từ hai con (đã tính theo khoảng cách tới `u`), cộng `Σ L[i]·R[j]` (i+j ≤ D) vào đáp án, rồi trả cho cha mảng `L + R` dịch phải 1 ô.

**Invariant:** hàm `dfs(u)` trả về đúng phân bố khoảng cách (≤ D) từ `cha của u` tới các lá trong cây con `u`, và biến toàn cục `ans` đã cộng đủ mọi cặp tốt có LCA nằm trong cây con `u`.

## 7. Các cách tiếp theo

### Cách 2 — DFS trả về danh sách khoảng cách của từng lá

Bước trung gian dễ hiểu: mỗi nút trả về **danh sách** khoảng cách từ nút đó tới từng lá bên dưới (chỉ giữ những khoảng cách `< D`). Tại nút trong, ghép mọi cặp (một từ trái, một từ phải).

In [11]:
def dfs_lists(root, distance):
    ans = 0

    def dfs(u):
        """Trả về list khoảng cách từ u tới các lá trong cây con u."""
        nonlocal ans
        if u is None:
            return []
        if u.left is None and u.right is None:
            return [0]
        left = [d + 1 for d in dfs(u.left)]      # khoảng cách tới u
        right = [d + 1 for d in dfs(u.right)]
        for a in left:
            for b in right:
                if a + b <= distance:
                    ans += 1
        return [d for d in left + right if d < distance]   # >= D thì vô dụng cho tổ tiên

    dfs(root)
    return ans

for vals, d, want in examples:
    print(vals, d, "->", dfs_lists(build_tree(vals), d), "mong đợi", want)

[1, 2, 3, None, 4] 3 -> 1 mong đợi 1
[1, 2, 3, 4, 5, 6, 7] 3 -> 2 mong đợi 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 -> 1 mong đợi 1


Độ dài danh sách tại một nút có thể tới số lá của cây con (`O(L)`), nên bước ghép cặp vẫn có thể là `O(L²)` tại gốc. Cải tiến tự nhiên: các lá **cùng khoảng cách** hoàn toàn thay thế được nhau → gom lại thành **mảng đếm**.

### Cách 3 — DFS trả mảng đếm lá theo độ sâu, `O(n · D²)`

Giải thích từng đoạn của `dfs(u)` (trả về `cnt` với `cnt[d]` = số lá cách **u** đúng `d` cạnh, `0 ≤ d ≤ D`):

1. `u is None` → mảng 0.
2. `u` là lá → `cnt[0] = 1`.
3. Lấy `L = dfs(u.left)`, `R = dfs(u.right)`; các khoảng cách này tính tới con, nên tới `u` là `+1`: lá ở `L[i]` cách `u` là `i + 1`.
4. Đếm cặp: `Σ L[i]·R[j]` với `(i+1) + (j+1) ≤ D`.
5. Mảng trả về: `res[d] = L[d−1] + R[d−1]` với `1 ≤ d ≤ D`.

In [12]:
def dfs_count(root, distance):
    D = distance
    ans = 0

    def dfs(u):
        nonlocal ans
        cnt = [0] * (D + 1)
        if u is None:
            return cnt
        if u.left is None and u.right is None:
            cnt[0] = 1
            return cnt
        L, R = dfs(u.left), dfs(u.right)
        for i in range(D + 1):
            if L[i] == 0:
                continue
            for j in range(D + 1):
                if (i + 1) + (j + 1) <= D:
                    ans += L[i] * R[j]
        for d in range(1, D + 1):
            cnt[d] = L[d - 1] + R[d - 1]
        return cnt

    dfs(root)
    return ans

for vals, d, want in examples:
    print(vals, d, "->", dfs_count(build_tree(vals), d), "mong đợi", want)

[1, 2, 3, None, 4] 3 -> 1 mong đợi 1
[1, 2, 3, 4, 5, 6, 7] 3 -> 2 mong đợi 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 -> 1 mong đợi 1


Code nộp LeetCode cho cách 3 (dùng quy ước "khoảng cách tới cha": lá trả `cnt[1] = 1`, nên tại nút trong điều kiện là `i + j ≤ D` và dịch mảng ngay khi trả về):

In [13]:
class SolutionCount:
    def countPairs(self, root: Optional[TreeNode], distance: int) -> int:
        self.ans = 0

        def dfs(u):
            # cnt[d] = số lá trong cây con u cách CHA của u đúng d cạnh
            cnt = [0] * (distance + 1)
            if u is None:
                return cnt
            if u.left is None and u.right is None:
                cnt[1] = 1
                return cnt
            L, R = dfs(u.left), dfs(u.right)
            for i in range(1, distance + 1):
                for j in range(1, distance + 1 - i):
                    self.ans += L[i] * R[j]
            for d in range(2, distance + 1):
                cnt[d] = L[d - 1] + R[d - 1]
            return cnt

        dfs(root)
        return self.ans

print([SolutionCount().countPairs(build_tree(v), d) for v, d, _ in examples])

[1, 2, 1]


**Độ phức tạp:** mỗi nút làm `O(D²)` việc → Time `O(n · D²)` (≤ `1024 · 100` ≈ 10⁵ phép); Space `O(n · D)` tổng cộng cho các mảng tạm, `O(h · D)` đồng thời trên ngăn xếp đệ quy.

### Cách 4 — Tối ưu gộp bằng tổng tiền tố, `O(n · D)`

Với mỗi `i`, ta cần `Σ_{j ≤ D − i} R[j]` — đó là **tổng tiền tố** của `R`. Tính `pre[k] = R[1] + … + R[k]` một lần (`O(D)`), rồi `ans += L[i] · pre[D − i]`. Với `D ≤ 10` lợi ích thực tế nhỏ, nhưng đây là kỹ thuật quan trọng khi `D` lớn.

In [14]:
class SolutionPrefix:
    def countPairs(self, root: Optional[TreeNode], distance: int) -> int:
        D = distance
        ans = 0

        def dfs(u):
            nonlocal ans
            cnt = [0] * (D + 1)
            if u is None:
                return cnt
            if u.left is None and u.right is None:
                cnt[1] = 1
                return cnt
            L, R = dfs(u.left), dfs(u.right)
            pre = [0] * (D + 1)
            for k in range(1, D + 1):
                pre[k] = pre[k - 1] + R[k]
            for i in range(1, D):
                ans += L[i] * pre[D - i]
            for d in range(2, D + 1):
                cnt[d] = L[d - 1] + R[d - 1]
            return cnt

        dfs(root)
        return ans

print([SolutionPrefix().countPairs(build_tree(v), d) for v, d, _ in examples])

[1, 2, 1]


**Độ phức tạp:** Time `O(n · D)`, Space như cách 3.

### Cách 5 — Lặp không đệ quy (post-order bằng stack)

Cây có tới 1024 nút; nếu cây là một "chuỗi" thì độ sâu đệ quy ~1024 — gần giới hạn mặc định của Python (1000). LeetCode tăng giới hạn nên bản đệ quy vẫn qua, nhưng bản lặp an toàn hơn: duyệt theo thứ tự "gốc–phải–trái" rồi đảo ngược để có post-order, lưu mảng đếm của mỗi nút vào `dict`.

In [15]:
class SolutionIter:
    def countPairs(self, root: Optional[TreeNode], distance: int) -> int:
        D = distance
        order, stack = [], [root]
        while stack:                       # thứ tự gốc trước con
            u = stack.pop()
            order.append(u)
            for v in (u.left, u.right):
                if v is not None:
                    stack.append(v)
        zero = [0] * (D + 1)
        memo, ans = {}, 0
        for u in reversed(order):          # con luôn được xử lý trước cha
            cnt = [0] * (D + 1)
            if u.left is None and u.right is None:
                cnt[1] = 1
            else:
                L = memo.pop(u.left, zero) if u.left else zero
                R = memo.pop(u.right, zero) if u.right else zero
                for i in range(1, D):
                    if L[i]:
                        ans += L[i] * sum(R[1:D - i + 1])
                for d in range(2, D + 1):
                    cnt[d] = L[d - 1] + R[d - 1]
            memo[u] = cnt
        return ans

print([SolutionIter().countPairs(build_tree(v), d) for v, d, _ in examples])

[1, 2, 1]


## 8. TRACE từng bước

Trace cách 3 (quy ước "khoảng cách tới cha") trên ví dụ 3, `D = 3`. Đọc mảng level-order `[7,1,4,6,null,5,3,null,null,null,null,null,2]` theo hàng đợi: `7 → (1, 4)`, `1 → (6, null)`, `4 → (5, 3)`, `6 → (null, null)`, `5 → (null, null)`, `3 → (null, 2)`. Vậy nút **2 là con phải của 3** (dễ nhầm là con của 6!):

```
        7
      /   \
     1     4
    /     / \
   6     5   3
              \
               2
```

Lá là `6, 5, 2`. Thứ tự post-order: 6, 1, 5, 2, 3, 4, 7.

| Nút | Loại | L (d=1..3) | R (d=1..3) | Cặp thêm | cnt trả về (d=1..3) |
|-----|------|------------|------------|----------|----------------------|
| 6 | lá | – | – | 0 | [1,0,0] |
| 1 | trong | [1,0,0] | [0,0,0] | 0 (một bên rỗng) | [0,1,0] |
| 5 | lá | – | – | 0 | [1,0,0] |
| 2 | lá | – | – | 0 | [1,0,0] |
| 3 | trong | [0,0,0] | [1,0,0] | 0 | [0,1,0] |
| 4 | trong | [1,0,0] | [0,1,0] | `L[1]·R[2]` (1+2 ≤ 3) = **1** | [0,1,1] |
| 7 | trong | [0,1,0] | [0,1,1] | `L[2]·R[1] = 0`; các cặp khác > 3 | – |

Tổng = 1, đúng là cặp `(5, 2)`: `5 → 4 → 3 → 2`. Hàm `trace_count` in lại đúng các dòng trên (thụt lề theo độ sâu đệ quy).

In [16]:
def trace_count(root, distance):
    D = distance
    ans = 0

    def dfs(u, depth=0):
        nonlocal ans
        cnt = [0] * (D + 1)
        if u is None:
            return cnt
        pad = "  " * depth
        if u.left is None and u.right is None:
            cnt[1] = 1
            print(f"{pad}nút {u.val}: lá -> trả {cnt[1:]}")
            return cnt
        L, R = dfs(u.left, depth + 1), dfs(u.right, depth + 1)
        add = 0
        for i in range(1, D + 1):
            for j in range(1, D + 1 - i):
                add += L[i] * R[j]
        ans += add
        for d in range(2, D + 1):
            cnt[d] = L[d - 1] + R[d - 1]
        print(f"{pad}nút {u.val}: L={L[1:]} R={R[1:]} cặp thêm={add} ans={ans} -> trả {cnt[1:]}")
        return cnt

    dfs(root)
    return ans

print("Kết quả:", trace_count(build_tree(examples[2][0]), 3))

    nút 6: lá -> trả [1, 0, 0]
  nút 1: L=[1, 0, 0] R=[0, 0, 0] cặp thêm=0 ans=0 -> trả [0, 1, 0]
    nút 5: lá -> trả [1, 0, 0]
      nút 2: lá -> trả [1, 0, 0]
    nút 3: L=[0, 0, 0] R=[1, 0, 0] cặp thêm=0 ans=0 -> trả [0, 1, 0]
  nút 4: L=[1, 0, 0] R=[0, 1, 0] cặp thêm=1 ans=1 -> trả [0, 1, 1]
nút 7: L=[0, 1, 0] R=[0, 1, 1] cặp thêm=0 ans=1 -> trả [0, 0, 2]
Kết quả: 1


Kiểm tra chéo bằng LCA: in cây và khoảng cách mọi cặp lá. Bài học: **luôn in cây** (hàm `show`) thay vì tự vẽ theo cảm giác — vẽ sai cây là lỗi rất hay gặp khi đọc mảng level-order có nhiều `null`.

In [17]:
show(build_tree(examples[2][0]))
r3 = build_tree(examples[2][0])
p3, d3 = parent_depth(r3)
lv3 = [u for u in d3 if u.left is None and u.right is None]
print("các lá:", [u.val for u in lv3])
for i in range(len(lv3)):
    for j in range(i + 1, len(lv3)):
        print(f"dist({lv3[i].val},{lv3[j].val}) =", dist_by_lca(lv3[i], lv3[j], p3, d3))

            2 (lá)
        3
    4
        5 (lá)
7
    1
        6 (lá)
các lá: [5, 2, 6]
dist(5,2) = 3
dist(5,6) = 4
dist(2,6) = 5


## 9. Code nộp LeetCode

Bản nộp chính: cách 3 (DFS mảng đếm), đúng chữ ký `countPairs(self, root, distance) -> int`. Trên LeetCode lớp `TreeNode` đã có sẵn; ở đây ta dùng lớp tự dựng.

In [18]:
class Solution:
    def countPairs(self, root: Optional[TreeNode], distance: int) -> int:
        ans = 0

        def dfs(u):
            # trả về cnt: cnt[d] = số lá trong cây con u cách CHA của u đúng d cạnh (1 <= d <= distance)
            nonlocal ans
            cnt = [0] * (distance + 1)
            if u is None:
                return cnt
            if u.left is None and u.right is None:
                cnt[1] = 1
                return cnt
            L, R = dfs(u.left), dfs(u.right)
            for i in range(1, distance):
                if L[i]:
                    for j in range(1, distance - i + 1):
                        ans += L[i] * R[j]
            for d in range(2, distance + 1):
                cnt[d] = L[d - 1] + R[d - 1]
            return cnt

        dfs(root)
        return ans

for vals, d, want in examples:
    got = Solution().countPairs(build_tree(vals), d)
    print(vals, d, "->", got, "OK" if got == want else "SAI")

[1, 2, 3, None, 4] 3 -> 1 OK
[1, 2, 3, 4, 5, 6, 7] 3 -> 2 OK
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 -> 1 OK


## 10. Vì sao thuật toán đúng

1. **Mỗi cặp lá có đúng một LCA.** Với hai lá khác nhau `a`, `b`, LCA `u` là nút trong (lá không thể là tổ tiên của nút khác), và `a`, `b` nằm ở hai cây con **khác nhau** của `u`. Đường đi ngắn nhất (duy nhất trong cây) có độ dài `dist(u,a) + dist(u,b)`.
2. **Tại `u` ta đếm đúng các cặp có LCA là `u`.** `L[i]·R[j]` đếm các cặp (lá trái ở khoảng cách `i`, lá phải ở khoảng cách `j`) — chính là mọi cặp có một lá bên trái, một lá bên phải. Điều kiện `i + j ≤ D` đúng bằng điều kiện "tốt".
3. **Cắt bỏ khoảng cách > D không làm mất cặp.** Nếu lá `a` cách `u` hơn `D` cạnh thì với mọi tổ tiên `w` của `u`, `dist(w, a) > D`, nên mọi cặp chứa `a` có LCA ở trên `u` đều dài hơn `D`.
4. **Invariant theo quy nạp post-order:** giả sử `dfs` đúng cho hai con; mảng trả về `cnt[d] = L[d−1] + R[d−1]` liệt kê đúng các lá dưới `u` theo khoảng cách tới cha của `u` (thêm một cạnh `u–cha`). Trường hợp cơ sở (lá, `None`) hiển nhiên.

Cộng theo mọi nút `u` ⇒ mỗi cặp tốt được đếm đúng một lần.

## 11. Phân tích Time / Space

Gọi `n` = số nút (≤ 1024), `L` = số lá, `h` = chiều cao, `D = distance` (≤ 10).

| Cách | Time | Space |
|------|------|-------|
| 1. Mọi cặp + LCA | `O(n + L² · h)` | `O(n)` |
| 1b. BFS giới hạn từ mỗi lá | `O(L · min(n, 2^D))` | `O(n)` |
| 2. DFS trả list khoảng cách | `O(n · L)` worst (danh sách dài + ghép cặp) | `O(n · L)` |
| 3. DFS mảng đếm | `O(n · D²)` | `O(n · D)` (mảng tạm), stack `O(h)` |
| 4. Tổng tiền tố | `O(n · D)` | như trên |
| 5. Lặp post-order | `O(n · D²)` | `O(n · D)`, không đệ quy |

Worst case của cách 1 là cây gần đầy đủ (nhiều lá) hoặc cây cao; cách 3/4 không phụ thuộc hình dạng cây.

## 12. Bộ test

Ví dụ đề, edge case (một nút, cây chuỗi, `distance = 1`), đối chiếu mọi cách với brute force LCA trên cây ngẫu nhiên (seed cố định, `distance` 1..10), và một cây lớn 1024 nút.

In [19]:
def random_tree(n, rng):
    """Cây nhị phân ngẫu nhiên n nút: gắn mỗi nút mới vào một chỗ trống ngẫu nhiên."""
    root = TreeNode(rng.randint(1, 100))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        parent, side = slots[k]
        slots[k] = slots[-1]
        slots.pop()
        node = TreeNode(rng.randint(1, 100))
        setattr(parent, side, node)
        slots.append((node, "left"))
        slots.append((node, "right"))
    return root

def chain(n):
    root = cur = TreeNode(1)
    for _ in range(n - 1):
        cur.left = TreeNode(1)
        cur = cur.left
    return root

METHODS = {
    "bfs": bfs_from_leaves,
    "lists": dfs_lists,
    "count": dfs_count,
    "SolutionCount": lambda r, d: SolutionCount().countPairs(r, d),
    "SolutionPrefix": lambda r, d: SolutionPrefix().countPairs(r, d),
    "SolutionIter": lambda r, d: SolutionIter().countPairs(r, d),
    "Solution": lambda r, d: Solution().countPairs(r, d),
}
edge = [([1], 1, 0), ([1], 10, 0), ([1, 2], 3, 0), ([1, 2, 3], 1, 0), ([1, 2, 3], 2, 1),
        ([1, 1, 1], 2, 1), ([1, 2, 3, 4, 5, 6, 7], 4, 6), ([1, 2, 3, 4, 5, 6, 7], 2, 2)]
for vals, d, want in examples + edge:
    assert brute_lca(build_tree(vals), d) == want, (vals, d)
    for name, f in METHODS.items():
        got = f(build_tree(vals), d)
        assert got == want, (name, vals, d, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1530)
for _ in range(300):
    n = rng.randint(1, 120)
    root = random_tree(n, rng)
    d = rng.randint(1, 10)
    ref = brute_lca(root, d)
    for name, f in METHODS.items():
        assert f(root, d) == ref, (name, to_list(root), d)
print("300 cây ngẫu nhiên (seed 1530): mọi cách khớp brute force LCA")

300 cây ngẫu nhiên (seed 1530): mọi cách khớp brute force LCA


In [21]:
# Cây chuỗi (chỉ 1 lá) và cây đầy đủ 1023 nút + 1 nút
assert all(f(chain(30), 10) == 0 for f in METHODS.values())
full = build_tree(list(range(1, 1024)) + [1])
big_rand = random_tree(1024, random.Random(5))
for name in ["SolutionCount", "SolutionPrefix", "SolutionIter", "Solution"]:
    t0 = time.perf_counter()
    res = [METHODS[name](full, 10), METHODS[name](big_rand, 10)]
    print(f"{name:15s}: {res} ({time.perf_counter() - t0:.3f}s)")
t0 = time.perf_counter()
print("brute LCA      :", [brute_lca(full, 10), brute_lca(big_rand, 10)], f"({time.perf_counter() - t0:.3f}s)")

SolutionCount  : [7920, 2147] (0.004s)
SolutionPrefix : [7920, 2147] (0.002s)
SolutionIter   : [7920, 2147] (0.002s)
Solution       : [7920, 2147] (0.002s)


brute LCA      : [7920, 2147] (0.144s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ ghép cặp giữa hai **lá anh em** trực tiếp

Nghĩ rằng cặp tốt chỉ là hai lá chung cha. Sai vì hai lá có thể cách nhau qua nhiều mức (ví dụ 1 và 3).

In [22]:
def wrong_siblings(root, distance):
    ans = 0
    def go(u):
        nonlocal ans
        if u is None:
            return
        l, r = u.left, u.right
        if l and r and not (l.left or l.right) and not (r.left or r.right) and 2 <= distance:
            ans += 1                 # SAI: bỏ sót cặp có LCA cao hơn
        go(l); go(r)
    go(root)
    return ans

for vals, d, want in examples:
    print(vals, d, "| sai:", wrong_siblings(build_tree(vals), d), "| đúng:", want)

[1, 2, 3, None, 4] 3 | sai: 0 | đúng: 1
[1, 2, 3, 4, 5, 6, 7] 3 | sai: 2 | đúng: 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 | sai: 0 | đúng: 1


Sửa: đếm theo LCA với phân bố khoảng cách đầy đủ (hàm `Solution`).

### Lỗi 2 — BFS từ mỗi lá nhưng quên chia 2 (hoặc đếm cả chính nó)

Mỗi cặp `(a, b)` được tìm thấy từ `a` và từ `b`. Nếu không loại `u is s` thì còn đếm thêm cả chính lá xuất phát.

In [23]:
def wrong_bfs(root, distance):
    adj, stack, lv = defaultdict(list), [root], []
    while stack:
        u = stack.pop()
        if not (u.left or u.right):
            lv.append(u)
        for v in (u.left, u.right):
            if v:
                adj[u].append(v); adj[v].append(u); stack.append(v)
    total = 0
    for s in lv:
        seen, q = {s}, deque([(s, 0)])
        while q:
            u, d = q.popleft()
            if not (u.left or u.right):
                total += 1           # SAI: đếm cả s, và không chia 2
            if d < distance:
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v); q.append((v, d + 1))
    return total

for vals, d, want in examples:
    print(vals, d, "| sai:", wrong_bfs(build_tree(vals), d), "| đúng:", want)

[1, 2, 3, None, 4] 3 | sai: 4 | đúng: 1
[1, 2, 3, 4, 5, 6, 7] 3 | sai: 8 | đúng: 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 | sai: 5 | đúng: 1


Sửa: bỏ qua `u is s` và trả `total // 2` (hàm `bfs_from_leaves`).

### Lỗi 3 — Lệch một khi dịch mảng (quên cộng cạnh `u → cha`)

Trả về `L + R` mà **không dịch**: khoảng cách bị đếm thiếu 1 cạnh mỗi mức → nhận cả những cặp quá xa.

In [24]:
def wrong_no_shift(root, distance):
    ans = 0
    def dfs(u):
        nonlocal ans
        cnt = [0] * (distance + 2)
        if u is None:
            return cnt
        if not (u.left or u.right):
            cnt[1] = 1
            return cnt
        L, R = dfs(u.left), dfs(u.right)
        for i in range(1, distance + 1):
            for j in range(1, distance + 1 - i):
                ans += L[i] * R[j]
        for d in range(1, distance + 1):
            cnt[d] = L[d] + R[d]      # SAI: phải là L[d-1] + R[d-1]
        return cnt
    dfs(root)
    return ans

for vals, d, want in examples + [([1, 2, 3, 4, 5, 6, 7], 3, 2)]:
    print(vals, d, "| sai:", wrong_no_shift(build_tree(vals), d), "| đúng:", want)

[1, 2, 3, None, 4] 3 | sai: 1 | đúng: 1
[1, 2, 3, 4, 5, 6, 7] 3 | sai: 6 | đúng: 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 | sai: 3 | đúng: 1
[1, 2, 3, 4, 5, 6, 7] 3 | sai: 6 | đúng: 2


Sửa: `cnt[d] = L[d − 1] + R[d − 1]` với `d ≥ 2` (lá trả `cnt[1] = 1`).

### Lỗi 4 — Dùng `val` làm khoá định danh nút

Ví dụ gom `parent[node.val]`: giá trị trùng nhau (đề cho `1 ≤ val ≤ 100`, tới 1024 nút!) sẽ ghi đè lẫn nhau.

In [25]:
def wrong_val_key(root, distance):
    parent, depth = {root.val: None}, {root.val: 0}
    stack, lv = [root], []
    while stack:
        u = stack.pop()
        if not (u.left or u.right):
            lv.append(u.val)
        for v in (u.left, u.right):
            if v:
                parent[v.val] = u.val            # SAI: trùng val bị ghi đè
                depth[v.val] = depth[u.val] + 1
                stack.append(v)
    def dist(a, b):
        s = 0
        while depth[a] > depth[b]: a, s = parent[a], s + 1
        while depth[b] > depth[a]: b, s = parent[b], s + 1
        while a != b: a, b, s = parent[a], parent[b], s + 2
        return s
    return sum(dist(lv[i], lv[j]) <= distance for i in range(len(lv)) for j in range(i + 1, len(lv)))

vals = [1, 1, 1, 1, 1, 1, 1]
print("sai:", wrong_val_key(build_tree(vals), 2), "| đúng:", Solution().countPairs(build_tree(vals), 2))

sai: 6 | đúng: 2


Sửa: dùng chính đối tượng nút làm khoá (`parent[v] = u`) như `parent_depth`.

### Lỗi 5 — Điều kiện `i + j < distance` thay vì `≤`

Đề nói "less than or equal to". Viết `<` làm mất các cặp có độ dài **đúng bằng** `distance` (như cặp duy nhất của ví dụ 1).

In [26]:
def wrong_strict(root, distance):
    return Solution().countPairs(root, distance - 1) if distance > 1 else 0   # tương đương dùng '<'

for vals, d, want in examples:
    print(vals, d, "| sai:", wrong_strict(build_tree(vals), d), "| đúng:", want)

[1, 2, 3, None, 4] 3 | sai: 0 | đúng: 1
[1, 2, 3, 4, 5, 6, 7] 3 | sai: 2 | đúng: 2
[7, 1, 4, 6, None, 5, 3, None, None, None, None, None, 2] 3 | sai: 0 | đúng: 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm cặp nút trong cây thoả điều kiện về khoảng cách"** ⇒ đếm mỗi cặp tại **LCA**; mỗi nút trả về một "tóm tắt" của cây con (mảng đếm theo độ sâu).
- Ràng buộc nhỏ cho tham số (`distance ≤ 10`) là gợi ý: kích thước tóm tắt = `O(D)`, chi phí gộp `O(D²)`.
- Bài cây cần "khoảng cách giữa các nút bất kỳ" ⇒ cân nhắc chuyển thành **đồ thị vô hướng** (thêm cạnh lên cha) rồi BFS.
- Luôn tự viết `build_tree` + `show` để kiểm tra cây đúng như đề trước khi tính tay.
- Bài cùng họ: *All Nodes Distance K in Binary Tree* (863), *Diameter of Binary Tree* (543), *Sum of Distances in Tree* (834), *Count Subtrees With Max Distance Between Cities* (1617).

## 15. Biến thể của bài

1. **Đếm cặp có khoảng cách đúng bằng `D`**: chỉ ghép `L[i]·R[j]` với `i + j == D`.
2. **Cặp nút bất kỳ (không chỉ lá)**: mỗi nút cũng đóng góp `cnt[0] = 1` của chính nó, và ghép thêm "nút `u` với các nút bên dưới".
3. **Cây tổng quát (n-ary)**: gộp lần lượt từng con vào mảng tích luỹ: trước khi cộng con mới, ghép nó với tích luỹ hiện tại (tránh đếm trùng).
4. **Có trọng số cạnh**: khoảng cách không còn nguyên nhỏ → dùng danh sách đã sắp xếp + hai con trỏ / small-to-large merging, hoặc centroid decomposition.
5. **`D` rất lớn**: dùng tổng tiền tố (cách 4) và kỹ thuật gộp "con nhỏ vào con lớn" để đạt `O(n log n)`.

In [27]:
def variant_exact(root, D):
    """Biến thể 1: số cặp lá cách nhau đúng D."""
    ans = 0
    def dfs(u):
        nonlocal ans
        cnt = [0] * (D + 1)
        if u is None:
            return cnt
        if not (u.left or u.right):
            cnt[1] = 1
            return cnt
        L, R = dfs(u.left), dfs(u.right)
        for i in range(1, D):
            ans += L[i] * R[D - i]
        for d in range(2, D + 1):
            cnt[d] = L[d - 1] + R[d - 1]
        return cnt
    dfs(root)
    return ans

r2 = build_tree([1, 2, 3, 4, 5, 6, 7])
print([variant_exact(r2, D) for D in range(1, 6)], "(kỳ vọng: 2 cặp dài 2, 4 cặp dài 4)")

[0, 2, 0, 4, 0] (kỳ vọng: 2 cặp dài 2, 4 cặp dài 4)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Mọi cặp + LCA | leo lên tổ tiên chung | `O(L²·h)` | `O(n)` | baseline, kiểm chứng |
| 1b. BFS từ mỗi lá | đồ thị vô hướng, BFS ≤ D bước | `O(L·min(n,2^D))` | `O(n)` | theo gợi ý đề, dễ nghĩ |
| 2. DFS trả list | ghép từng cặp khoảng cách | `O(n·L)` | `O(n·L)` | bước trung gian dễ hiểu |
| 3. DFS mảng đếm | gom lá cùng độ sâu, đếm tại LCA | `O(n·D²)` | `O(n·D)` | **bản nộp chính** |
| 4. Tổng tiền tố | `Σ R[j≤D−i]` bằng prefix | `O(n·D)` | `O(n·D)` | khi `D` lớn |
| 5. Lặp post-order | như 3, không đệ quy | `O(n·D²)` | `O(n·D)` | cây rất sâu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `root = [1,1,1]`, `distance = 2` — có bao nhiêu cặp lá tốt? Còn `distance = 1`?

**Bài 2 (tính tay):** Với cây `[1,2,3,4,5,6,7]`, `D = 4`, mảng `cnt` (quy ước tới cha, `d = 1..4`) mà nút `2` trả về cho gốc là gì? Tại gốc cộng thêm bao nhiêu cặp?

**Bài 3 (điền TODO):** Hoàn thiện `ex3_merge(L, R, D)` trả về số cặp `Σ L[i]·R[j]` với `i, j ≥ 1` và `i + j ≤ D`.

**Bài 4 (tự viết lại):** Viết `ex4_leaf_depths(root)` trả về danh sách độ sâu của mọi lá (gốc có độ sâu 0), theo thứ tự trái → phải.

**Bài 5 (mini challenge):** Viết `ex5_hist(root, D)` trả về mảng `h` với `h[k]` = số cặp lá có khoảng cách **đúng** `k` (`1 ≤ k ≤ D`). Kiểm tra: `sum(h[1:D+1]) == countPairs(root, D)`.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_merge(L: List[int], R: List[int], D: int) -> int:
    total = 0
    # TODO: hai vòng lặp i, j bắt đầu từ 1, điều kiện i + j <= D
    return total

In [29]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_leaf_depths(root: Optional[TreeNode]) -> List[int]:
    # TODO
    raise NotImplementedError

def ex5_hist(root: Optional[TreeNode], D: int) -> List[int]:
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Liệt kê lá. (2) Hai lá chung cha cách nhau 2 cạnh. (3) `distance = 2` → 1; `distance = 1` → 0 (hai lá khác nhau không bao giờ kề nhau).

**Bài 2.** (1) Lá 4 và 5 cách nút 2 một cạnh, cách gốc hai cạnh. (2) `cnt[2] = 2`, còn lại 0. (3) Tại gốc: `L[2]·R[2] = 2·2 = 4` với `2 + 2 ≤ 4`.

**Bài 3.** (1) `for i in range(1, D)`. (2) `for j in range(1, D − i + 1)`. (3) `total += L[i] * R[j]`.

**Bài 4.** (1) DFS mang theo độ sâu. (2) Gặp lá thì `append(depth)`. (3) Duyệt trái trước phải để giữ thứ tự.

**Bài 5.** (1) Giống `Solution` nhưng thay vì cộng vào một số, cộng vào `h[i + j]`. (2) `h` cần độ dài `D + 1`. (3) Đối chiếu với brute force LCA tính khoảng cách chính xác từng cặp.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
a1 = (Solution().countPairs(build_tree([1, 1, 1]), 2), Solution().countPairs(build_tree([1, 1, 1]), 1))
print("Bài 1:", a1)
assert a1 == (1, 0)

# Bài 2
D = 4
r = build_tree([1, 2, 3, 4, 5, 6, 7])
cnt_node2 = [0, 0, 2, 0, 0]          # index 0 không dùng
added_at_root = cnt_node2[2] * cnt_node2[2]
print("Bài 2: cnt của nút 2 =", cnt_node2[1:], "| cặp thêm tại gốc =", added_at_root)
assert added_at_root == 4 and Solution().countPairs(r, 4) == 2 + 4

Bài 1: (1, 0)
Bài 2: cnt của nút 2 = [0, 2, 0, 0] | cặp thêm tại gốc = 4


In [31]:
# Bài 3
def ex3_merge_sol(L, R, D):
    total = 0
    for i in range(1, D):
        for j in range(1, D - i + 1):
            total += L[i] * R[j]
    return total

assert ex3_merge_sol([0, 1, 0, 0], [0, 1, 0, 0], 3) == 1
assert ex3_merge_sol([0, 0, 2, 0, 0], [0, 0, 2, 0, 0], 4) == 4
assert ex3_merge_sol([0, 0, 2, 0], [0, 0, 2, 0], 3) == 0
print("Bài 3: OK")

Bài 3: OK


In [32]:
# Bài 4
def ex4_leaf_depths_sol(root):
    out = []
    def go(u, d):
        if u is None:
            return
        if not (u.left or u.right):
            out.append(d)
            return
        go(u.left, d + 1)
        go(u.right, d + 1)
    go(root, 0)
    return out

assert ex4_leaf_depths_sol(build_tree([1, 2, 3, None, 4])) == [2, 1]
assert ex4_leaf_depths_sol(build_tree(examples[2][0])) == [2, 2, 3]
assert ex4_leaf_depths_sol(build_tree([5])) == [0]
print("Bài 4: OK")

Bài 4: OK


In [33]:
# Bài 5
def ex5_hist_sol(root, D):
    h = [0] * (D + 1)
    def dfs(u):
        cnt = [0] * (D + 1)
        if u is None:
            return cnt
        if not (u.left or u.right):
            cnt[1] = 1
            return cnt
        L, R = dfs(u.left), dfs(u.right)
        for i in range(1, D):
            for j in range(1, D - i + 1):
                h[i + j] += L[i] * R[j]
        for d in range(2, D + 1):
            cnt[d] = L[d - 1] + R[d - 1]
        return cnt
    dfs(root)
    return h

def ex5_brute(root, D):
    par, dep = parent_depth(root)
    lv = [u for u in dep if not (u.left or u.right)]
    h = [0] * (D + 1)
    for i in range(len(lv)):
        for j in range(i + 1, len(lv)):
            k = dist_by_lca(lv[i], lv[j], par, dep)
            if k <= D:
                h[k] += 1
    return h

rng = random.Random(55)
for _ in range(150):
    root = random_tree(rng.randint(1, 80), rng)
    D = rng.randint(1, 10)
    h = ex5_hist_sol(root, D)
    assert h == ex5_brute(root, D) and sum(h) == Solution().countPairs(root, D)
print("Bài 5: OK, ví dụ", ex5_hist_sol(build_tree([1, 2, 3, 4, 5, 6, 7]), 4))

Bài 5: OK, ví dụ [0, 0, 2, 0, 4]


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def countPairs(self, root, distance):
        ans = 0
        def dfs(u):                       # cnt[d]: số lá cách CHA của u đúng d cạnh
            nonlocal ans
            cnt = [0] * (distance + 1)
            if not u: return cnt
            if not u.left and not u.right:
                cnt[1] = 1; return cnt
            L, R = dfs(u.left), dfs(u.right)
            for i in range(1, distance):
                for j in range(1, distance - i + 1):
                    ans += L[i] * R[j]    # cặp có LCA = u
            for d in range(2, distance + 1):
                cnt[d] = L[d - 1] + R[d - 1]
            return cnt
        dfs(root)
        return ans
```

**Template "DP trên cây đếm cặp theo khoảng cách":** post-order; mỗi nút trả mảng đếm theo độ sâu (cắt ở `D`); ghép hai con tại nút = đếm cặp có LCA ở đó; dịch mảng 1 ô khi trả lên cha.

**Checklist trước khi submit:**
- [ ] Lá trả `cnt[1] = 1` (hoặc `cnt[0] = 1` nếu quy ước tới chính nó) — thống nhất quy ước ở cả điều kiện ghép.
- [ ] Điều kiện ghép là `i + j ≤ distance` (có dấu bằng).
- [ ] Dịch mảng: `cnt[d] = L[d−1] + R[d−1]`, bỏ phần vượt `distance`.
- [ ] Không dùng `node.val` làm khoá.
- [ ] Cây một nút → 0; `distance = 1` → luôn 0.